In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np

import os
if "original_dir" not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, "../../.."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "checkpoints")

with open("data/traces_dataset/nonograms/epic.jsonl", "r") as f:
    json_dataset = [json.loads(line) for line in f.readlines()]
traces = [
    {"puzzle_idx": i, "method": m, **run}
    for i, p in enumerate(json_dataset)
    for m, d in p["traces"].items()
    for run in d
]

df = pd.json_normalize(traces)
df.head()


In [ ]:
pct = pd.crosstab(df["method"], df["stopping_reason"], normalize="index")

pct = pct.reindex(columns=["SOLVED", "NO_CHANGE", "TIMEOUT"], fill_value=0)
colors = {"SOLVED": "#4caf50", "NO_CHANGE": "#ff9800", "TIMEOUT": "#e53935"}

ax = pct.plot(kind="barh", stacked=True,
              color=[colors[c] for c in pct.columns],
              figsize=(8, 0.5 * len(pct) + 2),
              width=0.8)

ax.set_xlabel("Fraction of runs")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="stopping_reason", bbox_to_anchor=(1.02, 1), loc="upper left")

# Annotate percentages inside each segment
for i, method in enumerate(pct.index):
    left = 0
    for reason in pct.columns:
        val = pct.loc[method, reason]
        if val > 0.03:  # skip tiny labels
            ax.text(left + val / 2, i, f"{val:.0%}",
                    ha="center", va="center", color="white", fontsize=9)
        left += val

plt.tight_layout()
plt.show()

In [ ]:
def display_dataframe_as_pivot(df, values):
    pivot = df.pivot_table(index="method", columns="stopping_reason", values=values)
    cmap = plt.get_cmap("viridis")

    vals = pivot.values.astype(float)
    lo, hi = np.nanmin(vals), np.nanmax(vals)
    span = hi - lo if hi > lo else 1.0  # avoid div-by-zero

    def colour_cell(v):
        if pd.isna(v):
            return ""
        t = (v - lo) / span
        t = min(max(t, 0.0), 1.0)  # clamp
        r, g, b, _ = cmap(t)
        return f"background-color: rgba({r*255:.0f},{g*255:.0f},{b*255:.0f},0.8)"

    styled = (
        pivot.style
        .map(colour_cell)          # use .applymap if pandas < 2.1
        .format("{:.4f}")
        .set_caption(f"Pivot table of {values} by method and stopping_reason")
    )
    display(styled)

In [ ]:
mse_df = df.copy()
mse_df["final_mse"] = df["mse_losses"].map(lambda losses: losses[-1])
mse_df = (
    mse_df.groupby(["method", "stopping_reason"])
    .agg(MSE=("final_mse", "mean"))
    .reset_index()
)
display_dataframe_as_pivot(mse_df, "MSE")

In [ ]:
cr_df = df.copy()
cr_df["final_cr"] = df["cr_losses"].map(lambda losses: losses[-1])
cr_df = (
    cr_df.groupby(["method", "stopping_reason"])
    .agg(CR=("final_cr", "mean"))
    .reset_index()
)
display_dataframe_as_pivot(cr_df, "CR")

In [ ]:
mse_df = df.copy()
mse_df["final_mse"] = df["mse_losses"].map(lambda losses: losses[-1])
mse_df = (
    mse_df.groupby(["method", "stopping_reason"])
    .agg(MSE=("final_mse", "mean"))
    .reset_index()
)

cr_df = df.copy()
cr_df["final_cr"] = df["cr_losses"].map(lambda losses: losses[-1])
cr_df = (
    cr_df.groupby(["method", "stopping_reason"])
    .agg(CR=("final_cr", "mean"))
    .reset_index()
)

diff_df = mse_df.merge(cr_df, on=["method", "stopping_reason"])
diff_df["MSE - CR"] = diff_df["MSE"] - diff_df["CR"]

display_dataframe_as_pivot(diff_df, "MSE - CR")

In [ ]:
def loss_curves_with_iqr(df, loss_type="mse_losses"):
    fig, ax = plt.subplots(figsize=(9, 6))

    for method, group in df.groupby('method'):
        lists = group[loss_type].tolist()
        max_len = max(len(l) for l in lists)

        padded = np.empty((len(lists), max_len))
        padded[:] = np.nan
        for i, l in enumerate(lists):
            padded[i, :len(l)] = l

        # Forward-fill NaN values
        padded_ffill = pd.DataFrame(padded).ffill(axis=1).values

        median_loss = np.nanmedian(padded_ffill, axis=0)
        q1_loss = np.nanpercentile(padded_ffill, 25, axis=0)
        q3_loss = np.nanpercentile(padded_ffill, 75, axis=0)

        ax.plot(median_loss, label=method)
        ax.fill_between(range(len(median_loss)), q1_loss, q3_loss, alpha=0.2)

    ax.set_xscale('log')
    ax.set_xlabel('Step')
    ax.set_ylabel(f'{loss_type.upper()} Loss')
    ax.set_title(f'{loss_type.upper()} Loss by Method over Optimization Steps (SOLVED runs, forward-filled)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
loss_curves_with_iqr(df[df["stopping_reason"] == "SOLVED"], loss_type="mse_losses")

In [ ]:
loss_curves_with_iqr(df[df["stopping_reason"] == "SOLVED"], loss_type="cr_losses")